In [3]:
# scripts/train_simple_mil.py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, accuracy_score
import torchaudio
import random

# ----------------------- 配置 -----------------------
ROOT = Path(r"D:\Project_Github\audio_click_mil")
BAG_CSV = ROOT / "processed_data" / "balanced_bag_labels.csv"
INSTANCES_DIR = ROOT / "processed_data" / "instances"
FEATURE_CACHE = ROOT / "processed_data" / "simple_instance_features.pt"

OUTPUT_MODEL_DIR = ROOT / "processed_data" / "simple_mil_models"
OUTPUT_MODEL_DIR.mkdir(exist_ok=True, parents=True)

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-3
ATTN_THRESHOLD = 0.5

# 文献损失函数超参数
GAMMA = 2.0
LABEL_SMOOTH = 0.1
LAMBDA_TEMP = 0.1
LAMBDA_SPARSITY = 0.01
LAMBDA_INST = 0.001

# ----------------------- 工具函数 -----------------------
def extract_mfcc_features(waveform: np.ndarray) -> np.ndarray:
    """MFCC+delta+delta-delta → 时间均值 → 39维"""
    waveform = torch.from_numpy(waveform).unsqueeze(0).float()
    mfcc = torchaudio.transforms.MFCC(
        sample_rate=200000,
        n_mfcc=13,
        log_mels=True,
        melkwargs={"n_fft": 2048, "hop_length": 512}
    )(waveform)
    mfcc = mfcc.squeeze(0)
    
    deltas = torchaudio.functional.compute_deltas(mfcc)
    delta_deltas = torchaudio.functional.compute_deltas(deltas)
    
    feats = torch.cat([mfcc, deltas, delta_deltas], dim=0).T  # (T, 39)
    return feats.mean(dim=0).numpy().astype(np.float32)


class MILDataset(Dataset):
    def __init__(self, bag_df: pd.DataFrame, features: torch.Tensor):
        self.bag_df = bag_df.reset_index(drop=True)
        self.features = features  # (N_bags, 60, 39)

    def __len__(self):
        return len(self.bag_df)

    def __getitem__(self, idx):
        row = self.bag_df.iloc[idx]
        feat = self.features[idx]
        label = torch.tensor(row['bag_label'], dtype=torch.float32)
        return torch.tensor(feat, dtype=torch.float32), label, idx


class SimpleAttentionMIL(nn.Module):
    def __init__(self, feature_dim=39):
        super().__init__()
        self.instance_net = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        self.attn = nn.Linear(64, 1)
        self.classifier = nn.Linear(64, 1)

    def forward(self, bag):
        if len(bag.shape) != 3 or bag.shape[1] != 60 or bag.shape[2] != 39:
            raise ValueError(f"输入形状错误! 期望 (B, 60, 39), 实际得到 {bag.shape}")
        
        B, N, D = bag.shape
        inst_feat = self.instance_net(bag.view(-1, D)).view(B, N, 64)
        
        attn_logits = self.attn(inst_feat).squeeze(-1)
        attn_weights = torch.softmax(attn_logits, dim=1)
        
        bag_repr = torch.bmm(attn_weights.unsqueeze(1), inst_feat).squeeze(1)
        logit = self.classifier(bag_repr).squeeze(-1)
        return torch.sigmoid(logit), attn_weights, inst_feat   # 3个返回值


def compute_loss(preds, labels, attn_weights, inst_feats):
    """文献4组件损失"""
    labels_smooth = labels * (1 - LABEL_SMOOTH) + LABEL_SMOOTH / 2
    pt = torch.where(labels_smooth == 1, preds, 1 - preds)
    focal_weight = (1 - pt) ** GAMMA
    bce = nn.functional.binary_cross_entropy(preds, labels_smooth, reduction='none')
    loss_bce = (focal_weight * bce).mean()
    
    diff = attn_weights[:, 1:] - attn_weights[:, :-1]
    loss_temp = torch.mean(diff ** 2)
    
    loss_sparsity = torch.mean(torch.abs(attn_weights))
    
    inst_var = torch.var(inst_feats, dim=1).mean()
    loss_inst = inst_var * 2
    
    total_loss = (loss_bce +
                  LAMBDA_TEMP * loss_temp +
                  LAMBDA_SPARSITY * loss_sparsity +
                  LAMBDA_INST * loss_inst)
    return total_loss


def evaluate_localization(model, loader, bag_df, gt_centers_dict):
    model.eval()
    all_pred_centers = []
    all_gt_centers = []
    
    with torch.no_grad():
        for feats, _, idxs in tqdm(loader, desc="Localization eval", leave=False):
            feats = feats.to(DEVICE)
            preds, attn, _ = model(feats)          # ← 修复：接收3个返回值
            
            for b_idx, attn_w in zip(idxs, attn):
                row = bag_df.iloc[b_idx.item()]
                if row['bag_label'] == 0:
                    continue
                
                pred_mask = attn_w > ATTN_THRESHOLD
                pred_secs = np.where(pred_mask.cpu().numpy())[0]
                pred_centers = row['bag_start_sec'] + pred_secs + 0.5
                all_pred_centers.extend(pred_centers.tolist())
                
                gt_list = gt_centers_dict.get(int(row['file_num']), [])
                bag_gt = [c for c in gt_list 
                         if row['bag_start_sec'] <= c < row['bag_end_sec']]
                all_gt_centers.extend(bag_gt)
    
    def compute_metrics(tau):
        if not all_gt_centers or not all_pred_centers:
            return 1.0, 0.0, 0.0
        matches = sum(1 for g in all_gt_centers 
                     if any(abs(p - g) <= tau for p in all_pred_centers))
        recall = matches / len(all_gt_centers)
        precision = matches / len(all_pred_centers)
        fdr = 1 - precision
        f1 = 2 * recall * precision / (recall + precision) if (recall + precision) > 0 else 0
        return fdr, recall, f1

    results = {}
    for tau in [0.5, 1.0, 2.0]:
        fdr, rec, f1 = compute_metrics(tau)
        results[tau] = {"FDR": fdr, "Recall": rec, "F1": f1}
    return results


def main():
    print("加载 balanced_bag_labels.csv ...")
    bag_df = pd.read_csv(BAG_CSV)

    # GT centers
    click = pd.read_csv(ROOT / "data" / "ClickTrains.csv")
    burst = pd.read_csv(ROOT / "data" / "BurstPulseTrains.csv")
    buzz = pd.read_csv(ROOT / "data" / "BuzzTrains.csv")
    all_trains = pd.concat([click, burst, buzz], ignore_index=True)
    all_trains = all_trains[['Ori_file_num(No.)', 'Train_start(s)', 'Train_end(s)']]
    all_trains.columns = ['file_num', 'start', 'end']
    gt_centers = {}
    for fnum, g in all_trains.groupby('file_num'):
        centers = ((g['start'] + g['end']) / 2).values.tolist()
        gt_centers[int(fnum)] = centers

    # 特征缓存
    if FEATURE_CACHE.exists():
        features = torch.load(FEATURE_CACHE)
        print(f"找到缓存，形状: {features.shape}")
        if features.shape[-1] != 39 or features.shape[1] != 60:
            print("缓存形状错误，删除并重新提取...")
            FEATURE_CACHE.unlink()

    if not FEATURE_CACHE.exists():
        print("开始提取MFCC特征（每个实例时间均值 → 39维）...")
        features_list = []
        for _, row in tqdm(bag_df.iterrows(), total=len(bag_df), desc="Extracting MFCC"):
            npy_path = INSTANCES_DIR / f"file_{int(row['file_num']):02d}_bag_{int(row['bag_idx']):03d}_label_{int(row['bag_label'])}.npy"
            if not npy_path.exists():
                features_list.append(np.zeros((60, 39), dtype=np.float32))
                continue
            bag_wave = np.load(npy_path)
            bag_feat = np.stack([extract_mfcc_features(bag_wave[i]) for i in range(60)])
            features_list.append(bag_feat)
        
        features = torch.tensor(np.stack(features_list), dtype=torch.float32)
        torch.save(features, FEATURE_CACHE)
        print(f"特征提取完成，形状: {features.shape}")
    else:
        features = torch.load(FEATURE_CACHE)
        print(f"加载缓存成功，形状: {features.shape}")

    # 5折交叉验证
    print("\n开始5折交叉验证（使用文献4组件损失）...")
    fold_results = []
    
    for fold in range(5):
        print(f"\n=== Fold {fold+1}/5 ===")
        
        pos_idx = bag_df[bag_df['bag_label'] == 1].index.to_numpy().copy()
        neg_idx = bag_df[bag_df['bag_label'] == 0].index.to_numpy().copy()
        np.random.shuffle(pos_idx)
        np.random.shuffle(neg_idx)
        
        val_size = max(1, len(pos_idx) // 5)
        val_pos = pos_idx[:val_size]
        val_neg = neg_idx[:val_size]
        val_idx = np.concatenate([val_pos, val_neg])
        train_idx = np.setdiff1d(bag_df.index.to_numpy(), val_idx)

        train_df = bag_df.iloc[train_idx].reset_index(drop=True)
        val_df = bag_df.iloc[val_idx].reset_index(drop=True)

        train_ds = MILDataset(train_df, features[train_idx])
        val_ds = MILDataset(val_df, features[val_idx])
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        model = SimpleAttentionMIL().to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=LR)

        best_acc = 0.0
        for epoch in range(EPOCHS):
            model.train()
            for feats, labels, _ in tqdm(train_loader, desc=f"Fold{fold+1} Epoch {epoch+1}", leave=False):
                feats, labels = feats.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                preds, attn, inst_feat = model(feats)
                loss = compute_loss(preds, labels, attn, inst_feat)
                loss.backward()
                optimizer.step()

            model.eval()
            preds_all, labels_all = [], []
            with torch.no_grad():
                for feats, labels, _ in val_loader:
                    feats = feats.to(DEVICE)
                    preds, _, _ = model(feats)
                    preds_all.extend((preds > 0.5).cpu().numpy().astype(int))
                    labels_all.extend(labels.cpu().numpy().astype(int))
            
            acc = accuracy_score(labels_all, preds_all)
            if acc > best_acc:
                best_acc = acc
                torch.save(model.state_dict(), OUTPUT_MODEL_DIR / f"fold_{fold}.pth")

        # 最终评估
        model.load_state_dict(torch.load(OUTPUT_MODEL_DIR / f"fold_{fold}.pth"))
        model.eval()
        
        preds_all, labels_all = [], []
        with torch.no_grad():
            for feats, labels, _ in val_loader:
                feats = feats.to(DEVICE)
                preds, _, _ = model(feats)
                preds_all.extend((preds > 0.5).cpu().numpy().astype(int))
                labels_all.extend(labels.cpu().numpy().astype(int))

        p = precision_score(labels_all, preds_all, zero_division=0)
        r = recall_score(labels_all, preds_all, zero_division=0)
        a = accuracy_score(labels_all, preds_all)
        
        fold_results.append({"Precision": p, "Recall": r, "Accuracy": a})
        
        loc_res = evaluate_localization(model, val_loader, val_df, gt_centers)
        print(f"Fold {fold+1} Bag-level:  P={p:.4f}  R={r:.4f}  Acc={a:.4f}")
        print(f"Localization (τ=1.0s): FDR={loc_res[1.0]['FDR']:.4f}  Recall={loc_res[1.0]['Recall']:.4f}  F1={loc_res[1.0]['F1']:.4f}")

    df_res = pd.DataFrame(fold_results)
    print("\n=== 5折最终平均结果 ===")
    print(df_res.mean())
    print("\n标准差:")
    print(df_res.std())

    print("\n实验完成！模型已保存在 processed_data/simple_mil_models/")

if __name__ == "__main__":
    main()

加载 balanced_bag_labels.csv ...
开始提取MFCC特征（每个实例时间均值 → 39维）...


Extracting MFCC:   0%|          | 0/636 [00:00<?, ?it/s]d:\Python_env\Dolphin\Lib\site-packages\torchaudio\functional\functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (1025) may be set too low.
  warnings.warn(
Extracting MFCC: 100%|██████████| 636/636 [01:52<00:00,  5.65it/s]


特征提取完成，形状: torch.Size([636, 60, 39])

开始5折交叉验证（使用文献4组件损失）...

=== Fold 1/5 ===


Fold1 Epoch 1:   0%|          | 0/16 [00:00<?, ?it/s]C:\Users\Chico\AppData\Local\Temp\ipykernel_335912\913047900.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(feat, dtype=torch.float32), label, idx
Fold1 Epoch 2:   0%|          | 0/16 [00:00<?, ?it/s]C:\Users\Chico\AppData\Local\Temp\ipykernel_335912\913047900.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(feat, dtype=torch.float32), label, idx
Fold1 Epoch 3:   0%|          | 0/16 [00:00<?, ?it/s]          C:\Users\Chico\AppData\Local\Temp\ipykernel_335912\913047900.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTen

Fold 1 Bag-level:  P=0.0000  R=0.0000  Acc=0.5000
Localization (τ=1.0s): FDR=1.0000  Recall=0.0000  F1=0.0000

=== Fold 2/5 ===


...

Fold 3 Bag-level:  P=0.0000  R=0.0000  Acc=0.5000
Localization (τ=1.0s): FDR=1.0000  Recall=0.0000  F1=0.0000

=== Fold 4/5 ===


Fold4 Epoch 2:   0%|          | 0/16 [00:00<?, ?it/s]C:\Users\Chico\AppData\Local\Temp\ipykernel_335912\913047900.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(feat, dtype=torch.float32), label, idx
Fold4 Epoch 3:   0%|          | 0/16 [00:00<?, ?it/s]C:\Users\Chico\AppData\Local\Temp\ipykernel_335912\913047900.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(feat, dtype=torch.float32), label, idx
Fold4 Epoch 4:   0%|          | 0/16 [00:00<?, ?it/s]C:\Users\Chico\AppData\Local\Temp\ipykernel_335912\913047900.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach

Fold 4 Bag-level:  P=0.0000  R=0.0000  Acc=0.5000
Localization (τ=1.0s): FDR=1.0000  Recall=0.0000  F1=0.0000

=== Fold 5/5 ===


Fold5 Epoch 2:   0%|          | 0/16 [00:00<?, ?it/s]          C:\Users\Chico\AppData\Local\Temp\ipykernel_335912\913047900.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(feat, dtype=torch.float32), label, idx
Fold5 Epoch 3:   0%|          | 0/16 [00:00<?, ?it/s]C:\Users\Chico\AppData\Local\Temp\ipykernel_335912\913047900.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(feat, dtype=torch.float32), label, idx
Fold5 Epoch 4:   0%|          | 0/16 [00:00<?, ?it/s]C:\Users\Chico\AppData\Local\Temp\ipykernel_335912\913047900.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTen

Fold 5 Bag-level:  P=0.0000  R=0.0000  Acc=0.5000
Localization (τ=1.0s): FDR=1.0000  Recall=0.0000  F1=0.0000

=== 5折最终平均结果 ===
Precision    0.0
Recall       0.0
Accuracy     0.5
dtype: float64

标准差:
Precision    0.0
Recall       0.0
Accuracy     0.0
dtype: float64

实验完成！模型已保存在 processed_data/simple_mil_models/
